# Character N-Grams con LinearSVC

En este notebook implementamos el mejor enfoque encontrado para clasificar textos históricos en español por su década de origen. Se utilizan n-grams de caracteres como representación del texto, preservando toda la información ortográfica y morfológica del español histórico (siglos XVI–XIX). Como clasificador se usa una Máquina de Vectores de Soporte lineal (`LinearSVC`), que resulta especialmente eficiente para datos de alta dimensionalidad como los vectores TF-IDF de caracteres.

El pipeline encapsula el vectorizador y el clasificador dentro de un objeto único, garantizando que la transformación TF-IDF se ajuste únicamente con los datos de entrenamiento de cada fold durante la validación cruzada y evitando así cualquier fuga de información.

## 1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2. Carga de los datos

Cargamos el conjunto de entrenamiento y el de evaluación de forma independiente.

In [ ]:
df = pd.read_csv('train.csv')
df_eval = pd.read_csv('eval.csv')

df['text']      = df['text'].fillna('')
df_eval['text'] = df_eval['text'].fillna('')

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Exploración del conjunto de datos

Revisamos la distribución de la variable objetivo y la calidad general del conjunto.

In [ ]:
df['decade'].value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), title='Distribución de décadas'
)
plt.xlabel('Década')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
def reporte_calidad(df):
    reporte_de_cualidad = {
        'Total records': len(df),
        'duplicated record': df.duplicated().sum(),
        'missing values': df.isnull().sum().to_dict(),
        'data types': df.dtypes.astype(str).to_dict(),
    }
    return reporte_de_cualidad

print(reporte_calidad(df))

## 4. Preprocesamiento del texto

A diferencia de los enfoques basados en palabras, aquí no aplicamos limpieza manual del texto. La vectorización se basa en caracteres (`analyzer='char'`), por lo que preservar la ortografía original —incluyendo tildes, la ñ y otros diacríticos— es fundamental: estos caracteres evolucionaron de forma distinta en cada siglo y constituyen señales temporales directas.

La única normalización que realizamos es `lowercase=True` dentro del vectorizador. Se desactiva explícitamente el strip de acentos (`strip_accents=None`) para conservar toda esta información.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print('Registros tras eliminar duplicados:', len(df))
df[['text', 'decade']].head()

## 5. Partición de los datos

In [ ]:
X = df['text']
y = df['decade']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

Se usa `stratify=y` para mantener la proporción de clases en ambas particiones.

In [ ]:
X_train.shape, X_val.shape

## 6. Construcción del pipeline

El pipeline encadena el vectorizador TF-IDF de caracteres con el clasificador `LinearSVC`. Al estar incluido dentro del pipeline, el vectorizador se ajusta únicamente sobre los datos de entrenamiento en cada partición interna de la validación cruzada, lo que garantiza que no hay fuga de información del conjunto de validación hacia el vocabulario ni hacia los pesos IDF.

La configuración del vectorizador utiliza los hiperparámetros óptimos identificados en las iteraciones de experimentación: `ngram_range=(1,7)` captura desde caracteres individuales hasta secuencias de 7 caracteres, y `sublinear_tf=True` aplica escala logarítmica a las frecuencias de término para reducir el impacto de los n-grams muy frecuentes.

In [ ]:
pipeline_charsvm = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents=None,
        analyzer='char',
        ngram_range=(1, 7),
        sublinear_tf=True,
        max_features=300000,
        dtype=np.float32,
    )),
    ('clf', LinearSVC()),
])

## 7. Entrenamiento con búsqueda de hiperparámetros

La búsqueda explora el parámetro de regularización `C` del clasificador y la frecuencia mínima de documento `min_df` del vectorizador. Un `C` pequeño impone mayor regularización y reduce el sobreajuste; `min_df=1` incluye n-grams que aparecen una sola vez en el corpus, lo que puede capturar patrones ortográficos muy específicos de una época.

In [ ]:
param_grid = {
    'tfidf__min_df': [1, 2],
    'clf__C': [0.20, 0.30, 0.40, 0.50],
}

In [ ]:
kfold = KFold(n_splits=3, shuffle=True, random_state=0)
grid_charsvm = GridSearchCV(
    pipeline_charsvm, param_grid, cv=kfold, scoring='accuracy', n_jobs=-1, verbose=1
)

In [ ]:
grid_charsvm.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros:', grid_charsvm.best_params_)
print('Mejor score CV (accuracy):', round(grid_charsvm.best_score_, 4))

In [ ]:
resultados_cv = pd.DataFrame(grid_charsvm.cv_results_)
cols = ['param_clf__C', 'param_tfidf__min_df', 'mean_test_score', 'std_test_score', 'rank_test_score']
print(resultados_cv[cols].sort_values('rank_test_score').to_string(index=False))

## 8. Evaluación del mejor modelo

Aplicamos el mejor pipeline encontrado por GridSearchCV sobre el conjunto de validación.

In [ ]:
best_model = grid_charsvm.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

#### Comparación de rendimientos sobre entrenamiento y validación

In [ ]:
print('Accuracy en entrenamiento:', round(accuracy_score(y_train, y_pred_train), 4))
print('Accuracy en validación:   ', round(accuracy_score(y_val,   y_pred_val),   4))
print('Mejor score CV (accuracy):', round(grid_charsvm.best_score_,               4))

In [ ]:
print(classification_report(y_val, y_pred_val))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_val, ax=ax, colorbar=False)
plt.title('Matriz de confusión — validación')
plt.tight_layout()
plt.show()

## 9. Predicciones sobre eval.csv

Reentrenamos el mejor pipeline sobre el conjunto de entrenamiento completo antes de generar las predicciones finales. Esto aprovecha todos los datos disponibles y mejora la estimación del modelo.

In [ ]:
from sklearn.base import clone

final_model = clone(best_model)
final_model.fit(X, y)

print('Modelo reentrenado sobre el conjunto completo.')

In [ ]:
y_eval_pred = final_model.predict(df_eval['text']).astype(int)

submission = pd.DataFrame({'id': df_eval['id'], 'answer': y_eval_pred})
submission.to_csv('submission_char_svm.csv', index=False)

print('Filas generadas:', len(submission))
submission.head()